# MAVLink: talking to drones

The complete common dialect is generated from the official XML; a simulated quadcopter flies on an in-memory UDP link. Commands move real vehicles — keep to the simulator here.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.15.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.15.0-preview.1"

## Generated messages and a frame
Every message is a C# class with its id, CRC_EXTRA and wire layout.

In [ ]:
using IoTCom.Net.Protocols.Mavlink;
using IoTCom.Net.Protocols.Mavlink.Common;

Console.WriteLine($"{CommonDialect.Instance.Messages.Count} messages; HEARTBEAT crc_extra {Heartbeat.MavlinkCrcExtra}");
var hb = new Heartbeat { Type = MavType.Quadrotor, Autopilot = MavAutopilot.Px4, SystemStatus = MavState.Active };
var frame = MavlinkCodec.Encode(hb, sequence: 0, systemId: 1, componentId: 1);
Console.WriteLine(Convert.ToHexString(frame));
foreach (var f in MavlinkAnatomy.Describe(frame, CommonDialect.Instance)) Console.WriteLine($"{f.Name,-10} {f.Value}");

## Signing
A signed frame carries link id, timestamp and a 6-byte SHA-256 signature; a forged one is rejected.

In [ ]:
var key = MavlinkSigning.FromPassphrase("demo");
var signed = MavlinkCodec.Encode(hb, 1, 1, 1, signing: key);
var parser = new MavlinkParser(CommonDialect.Instance, MavlinkSigning.FromPassphrase("demo"));
parser.Feed(signed);
Console.WriteLine(parser.TryRead(out var ok) ? $"accepted, signed={ok.Signed}" : "rejected");
signed[^1] ^= 1;
parser.Feed(signed);
Console.WriteLine(parser.TryRead(out _) ? "accepted" : $"forged frame rejected ({parser.SignatureErrors})");

## Fly the simulator from a ground station

In [ ]:
using System.Net;
using IoTCom.Net.Transports;

var net = new InMemoryDatagramNetwork();
var gcsAddress = new IPEndPoint(IPAddress.Loopback, 14550);
await using var vehicle = MavlinkConnection.Create(o => { o.UseInMemory(net).SendTo(gcsAddress); (o.SystemId, o.ComponentId) = (1, 1); });
await using var sim = new MavlinkVehicleSimulator(vehicle);
await using var link = MavlinkConnection.Create(o => o.UseInMemory(net, gcsAddress));
using var gcs = new MavlinkGroundStation(link);
await vehicle.ConnectAsync(); await link.ConnectAsync(); sim.Start();
await gcs.WaitForHeartbeatAsync(TimeSpan.FromSeconds(5));
Console.WriteLine($"ARM → {await gcs.ArmAsync()}, TAKEOFF → {await gcs.TakeoffAsync(10)}");
await Task.Delay(4000);
Console.WriteLine($"altitude {gcs.State.RelativeAltitude:0.0} m, battery {gcs.State.BatteryVoltage:0.00} V, packets {link.Statistics.PacketsReceived}");
var p = await gcs.ReadParametersAsync();
Console.WriteLine(string.Join(", ", p.Take(4).Select(kv => $"{kv.Key}={kv.Value}")));

## Going further
The Gallery demo *Drone telemetry over MAVLink* adds an artificial horizon and a track map; `iotcom mavlink simulate` lets QGroundControl connect. See `docs/en/protocols/mavlink.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*